In [ ]:
from collections import Counter
from copy import deepcopy

import nibabel as nib
import numpy as np
import torch
from skimage import metrics
from torch.nn.parallel import DataParallel as DP
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from cfg import ModelCfg, TrainCfg
from data_reader import MBAS2024
from dataset import MBAS2024Dataset
from models import MultiHead_MBAS2024
from pipeline import run_pipeline
from trainer import MBAS2024Trainer
from utils.scoring_metrics import _hausdorff_distance, compute_challenge_metrics, hausdorff_distance

%load_ext autoreload
%autoreload 2

In [ ]:
db_dir = "/home/wenh06/Jupyter/wenhao/Hot-Data/MBAS2024/"

In [ ]:
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

# Stage 0

In [ ]:
train_config = deepcopy(TrainCfg)
train_config.db_dir = db_dir
train_config.debug = False
train_config.stage = 0
train_config.n_epochs = 100
train_config.apply_mclahe = True
train_config.batch_size = 1

In [ ]:
model_config = deepcopy(ModelCfg)
model_config.stage = train_config.stage
model_config.seg_model_name = "nestedvnet"  # "vnet", "nestedvnet"
model_config.apply_mclahe = train_config.apply_mclahe
model = MultiHead_MBAS2024(config=model_config)

In [ ]:
# if torch.cuda.device_count() > 1:
#     model = DP(model)
# model = DDP(model)
model = model.to(device=device)

In [ ]:
if isinstance(model, DP):
    print(model.module.module_size, model.module.module_size_)
else:
    print(model.module_size, model.module_size_)

In [ ]:
trainer = MBAS2024Trainer(
    model=model,
    model_config=model_config,
    train_config=train_config,
    device=device,
)

In [ ]:
best_model_state_dict = trainer.train()

In [ ]:
trainer.log_manager.flush()
trainer.log_manager.close()

In [ ]:
del trainer, model, best_model_state_dict
torch.cuda.empty_cache()

# Stage 1

In [ ]:
train_config = deepcopy(TrainCfg)
train_config.db_dir = db_dir
train_config.debug = False
train_config.stage = 1
train_config.n_epochs = 100
train_config.apply_mclahe = True
train_config.batch_size = 1

In [ ]:
model_config = deepcopy(ModelCfg)
model_config.stage = train_config.stage
model_config.seg_model_name = "nestedvnet"  # "vnet", "nestedvnet"
model_config.apply_mclahe = train_config.apply_mclahe
model = MultiHead_MBAS2024(config=model_config)

In [ ]:
# if torch.cuda.device_count() > 1:
#     model = DP(model)
# model = DDP(model)
model = model.to(device=device)

In [ ]:
trainer = MBAS2024Trainer(
    model=model,
    model_config=model_config,
    train_config=train_config,
    device=device,
)

In [ ]:
best_model_state_dict = trainer.train()

In [ ]:
trainer.log_manager.flush()
trainer.log_manager.close()

In [ ]:
del trainer, model, best_model_state_dict
torch.cuda.empty_cache()

# Inference

In [ ]:
stage0_model = MultiHead_MBAS2024.from_checkpoint("./saved_models/vnet/v2/stage0-model.pth.tar")[0].to(device).eval()
stage1_model = MultiHead_MBAS2024.from_checkpoint("./saved_models/vnet/v2/stage1-model.pth.tar")[0].to(device).eval()

In [ ]:
dr = MBAS2024(db_dir)

In [ ]:
image = dr.load_data(0)
ann = dr.load_ann(0)

In [ ]:
pred_mask = run_pipeline(image, stage0_model, stage1_model)
ann = ann.astype(pred_mask.dtype)

In [ ]:
compute_challenge_metrics([ann], [pred_mask])